# nb20 - Comparison plots (plotly)

The comparison figures Felipe/Carla asked for (2026-07-16), built from the per-cluster prediction CSVs written by nb19: resolution per architecture and dataset, vs true energy, vs ET, and by ECAL region, with the requested per-bin uncertainty 0.96 sigma_eff / sqrt(n). Project plotting standard is plotly (nb15-17); this revision replaces the earlier matplotlib version - the static PNG exports (fig1-fig5) remain in reports/figures/, interactive HTML copies land in reports/figures/interactive/. Test split only, seed-averaged predictions per model.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

REPO = Path(os.environ.get("REPO_DIR", "..")).resolve()
sys.path.insert(0, str(REPO / "scripts"))
from run_experiments import resolution
from plot_resolution import polish, export

pio.templates.default = "plotly_white"
PRED = REPO / "reports" / "predictions"
FIGDIR = REPO / "reports" / "figures" / "interactive"
FIGDIR.mkdir(parents=True, exist_ok=True)
H = 450
print("repo:", REPO)
print("figures:", FIGDIR)

repo: /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer
figures: /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive


Load every prediction CSV, keep `split=='test'`, verify the seed blocks share one event ordering, then average `pred_energy` across seeds.

In [2]:
BASE_MODELS = ["MeanDirect", "MeanResidual", "EfnResidual", "PairT", "Spacetime", "GateHuber", "BDT", "CalibratedSum"]
EXTRA = [("minbias", "SubNetW4"), ("minbias", "SubNetW4CleanAux"), ("minbias", "SubEnsembleTTA"),
         ("minbias", "SubNetW4CleanAuxQuant"), ("minbias", "SubNetW4CleanAuxQdEma")]

def load(dataset, model):
    f = PRED / f"{dataset}__{model}.csv"
    if not f.exists():
        return None
    df = pd.read_csv(f)
    df = df[df["split"] == "test"].reset_index(drop=True)
    blocks = [df[df["seed"] == s].reset_index(drop=True) for s in sorted(df["seed"].unique())]
    t0 = blocks[0]["true_energy"].to_numpy()
    assert all(np.allclose(b["true_energy"].to_numpy(), t0) for b in blocks)
    pred = np.mean([b["pred_energy"].to_numpy() for b in blocks], axis=0)
    b0 = blocks[0]
    return {"true": t0, "pred": pred, "region": b0["region_name"].to_numpy(), "ET": b0["ET"].to_numpy(), "n_seeds": len(blocks)}

data, missing = {}, []
for ds in ["clean", "minbias"]:
    for m in BASE_MODELS:
        r = load(ds, m)
        if r is None:
            missing.append(f"{ds}__{m}")
        else:
            data[(ds, m)] = r
for ds, m in EXTRA:
    r = load(ds, m)
    if r is None:
        missing.append(f"{ds}__{m}")
    else:
        data[(ds, m)] = r

for (ds, m), d in data.items():
    print(f"{ds:8s} {m:18s} n={len(d['true']):6d} seeds={d['n_seeds']}")
print("missing:", missing if missing else "none")

clean    MeanDirect         n=  5021 seeds=5
clean    MeanResidual       n=  5021 seeds=5
clean    EfnResidual        n=  5021 seeds=5
clean    PairT              n=  5021 seeds=5
clean    Spacetime          n=  5021 seeds=5
clean    GateHuber          n=  5021 seeds=5
clean    BDT                n=  5021 seeds=1
clean    CalibratedSum      n=  5021 seeds=1
minbias  MeanDirect         n= 12227 seeds=5
minbias  MeanResidual       n= 12227 seeds=5
minbias  EfnResidual        n= 12227 seeds=5
minbias  PairT              n= 12227 seeds=5
minbias  Spacetime          n= 12227 seeds=5
minbias  GateHuber          n= 12227 seeds=5
minbias  BDT                n= 12227 seeds=1
minbias  CalibratedSum      n= 12227 seeds=1
minbias  SubNetW4           n= 10884 seeds=5
minbias  SubNetW4CleanAux   n= 10884 seeds=2
minbias  SubEnsembleTTA     n= 10884 seeds=1
minbias  SubNetW4CleanAuxQuant n= 10884 seeds=5
minbias  SubNetW4CleanAuxQdEma n= 10884 seeds=5
missing: none


Aggregate `sigma_eff` per model and dataset, plus the shared quantile-bin helper (n >= 50 per bin).

In [3]:
rows = []
for (ds, m), d in data.items():
    rows.append({"model": m, "dataset": ds, "sigma_eff": resolution(d["pred"], d["true"])["sigma_eff"], "n": len(d["true"])})
summ = pd.DataFrame(rows).pivot(index="model", columns="dataset", values="sigma_eff").sort_values("minbias")
ORDER = list(summ.index)
print(summ.to_string())

def qedges(x, nbins=8):
    return np.unique(np.quantile(x, np.linspace(0, 1, nbins + 1)))

def qbin(x, true, pred, edges, min_n=50):
    idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)
    xs, ys, es = [], [], []
    for b in range(len(edges) - 1):
        sel = idx == b
        n = int(sel.sum())
        if n < min_n:
            continue
        s = resolution(pred[sel], true[sel])["sigma_eff"]
        xs.append(float(np.median(x[sel])))
        ys.append(s)
        es.append(0.96 * s / np.sqrt(n))
    return xs, ys, es

PALETTE = px.colors.qualitative.D3 + px.colors.qualitative.Set2
CMAP = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(ORDER)}

dataset                 clean  minbias
model                                 
SubNetW4CleanAuxQdEma     NaN   0.0409
SubNetW4CleanAuxQuant     NaN   0.0426
SubEnsembleTTA            NaN   0.0440
SubNetW4                  NaN   0.0446
SubNetW4CleanAux          NaN   0.0452
GateHuber              0.0443   0.0463
Spacetime              0.0417   0.0541
PairT                  0.0399   0.0624
EfnResidual            0.0400   0.0626
MeanDirect             0.0463   0.0648
MeanResidual           0.0397   0.0671
BDT                    0.0528   0.1253
CalibratedSum          0.0822   0.1837


## ifig1 · Summary bars

Aggregate `sigma_eff` per model, clean vs minbias, sorted by minbias.

In [4]:
fig1 = go.Figure()
for ds in ["minbias", "clean"]:
    ys = [summ.loc[m, ds] if ds in summ.columns else np.nan for m in ORDER]
    fig1.add_bar(name=ds, x=ORDER, y=ys,
                 text=[f"{v:.3f}" if np.isfinite(v) else "" for v in ys],
                 textposition="outside")
fig1.update_layout(barmode="group", height=H)
polish(fig1, "sigma_eff per model (test split, seed-averaged)", "model", "sigma_eff")
export(fig1, FIGDIR / "ifig1_summary_bars.html")
fig1.show()

wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig1_summary_bars.html


wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig1_summary_bars.png


## ifig2 · sigma_eff vs true energy (minbias)

8 quantile bins in true energy, x = bin median, error bars 0.96 sigma / sqrt(n).

In [5]:
ref_mb = data[("minbias", "GateHuber")]
edges_E_mb = qedges(ref_mb["true"])
fig2 = go.Figure()
for m in ORDER:
    d = data.get(("minbias", m))
    if d is None:
        continue
    xs, ys, es = qbin(d["true"], d["true"], d["pred"], edges_E_mb)
    fig2.add_scatter(x=xs, y=ys, error_y=dict(type="data", array=es), mode="lines+markers",
                     name=m, line=dict(color=CMAP[m]))
polish(fig2, "sigma_eff vs true energy (minbias)", "true energy [GeV]", "sigma_eff")
fig2.update_layout(height=H)
export(fig2, FIGDIR / "ifig2_vsE_by_arch.html")
fig2.show()

wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig2_vsE_by_arch.html
wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig2_vsE_by_arch.png


## ifig3 · Clean vs minbias

Best 3 minbias models plus CalibratedSum; solid = minbias, dashed = clean.

In [6]:
best3 = [m for m in ORDER if m != "CalibratedSum"][:3]
sel_models = best3 + ["CalibratedSum"]
edges_by_ds = {"minbias": edges_E_mb}
clean_keys = [k for k in data if k[0] == "clean"]
if clean_keys:
    edges_by_ds["clean"] = qedges(data[clean_keys[0]]["true"])
fig3 = go.Figure()
for m in sel_models:
    for ds, dash in [("minbias", "solid"), ("clean", "dash")]:
        d = data.get((ds, m))
        if d is None:
            continue
        xs, ys, es = qbin(d["true"], d["true"], d["pred"], edges_by_ds[ds])
        fig3.add_scatter(x=xs, y=ys, error_y=dict(type="data", array=es), mode="lines+markers",
                         name=f"{m} ({ds})", line=dict(color=CMAP[m], dash=dash))
polish(fig3, "Clean vs minbias: best 3 models + CalibratedSum", "true energy [GeV]", "sigma_eff")
fig3.update_layout(height=H)
export(fig3, FIGDIR / "ifig3_clean_vs_minbias.html")
fig3.show()

wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig3_clean_vs_minbias.html


wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig3_clean_vs_minbias.png


## ifig4 · By detector region (minbias)

Grouped bars per `region_name`, regions with n >= 200 only.

In [7]:
reg_counts = pd.Series(ref_mb["region"]).value_counts()
regions = sorted([r for r in reg_counts.index if reg_counts[r] >= 200], key=lambda r: int(r.replace("mm", "")))
fig4 = go.Figure()
for m in ORDER:
    d = data.get(("minbias", m))
    if d is None:
        continue
    ys = []
    for r in regions:
        sel = d["region"] == r
        ys.append(resolution(d["pred"][sel], d["true"][sel])["sigma_eff"] if sel.sum() >= 200 else np.nan)
    fig4.add_bar(name=m, x=regions, y=ys, marker_color=CMAP[m])
fig4.update_layout(barmode="group", height=H)
polish(fig4, "sigma_eff by cell-pitch region (minbias)", "region (cell pitch)", "sigma_eff")
export(fig4, FIGDIR / "ifig4_by_region.html")
fig4.show()

wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig4_by_region.html
wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig4_by_region.png


## ifig5 · sigma_eff vs ET (minbias)

Same binning scheme as ifig2, but in transverse energy.

In [8]:
edges_ET_mb = qedges(ref_mb["ET"])
fig5 = go.Figure()
for m in ORDER:
    d = data.get(("minbias", m))
    if d is None:
        continue
    xs, ys, es = qbin(d["ET"], d["true"], d["pred"], edges_ET_mb)
    fig5.add_scatter(x=xs, y=ys, error_y=dict(type="data", array=es), mode="lines+markers",
                     name=m, line=dict(color=CMAP[m]))
polish(fig5, "sigma_eff vs ET (minbias)", "E_T [GeV]", "sigma_eff")
fig5.update_layout(height=H)
export(fig5, FIGDIR / "ifig5_vsET_by_arch.html")
fig5.show()

wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig5_vsET_by_arch.html


wrote /home/lworakan/Documents/GitHub/GSoC2026-picocal-spacetime-transformer/reports/figures/interactive/ifig5_vsET_by_arch.png


Saved HTML files.

In [9]:
for f in sorted(FIGDIR.glob("ifig*.html")):
    print(f.name, f"{f.stat().st_size/1e6:.1f} MB")

ifig10_lowE_contamination.html 0.0 MB
ifig11_deblend_diag.html 0.1 MB
ifig1_summary_bars.html 0.0 MB
ifig2_vsE_by_arch.html 0.0 MB
ifig3_clean_vs_minbias.html 0.0 MB
ifig4_by_region.html 0.0 MB
ifig5_vsET_by_arch.html 0.0 MB
ifig6_request_tables.html 0.0 MB
ifig7_request_grid.html 0.0 MB
ifig8_request_spacetime.html 0.0 MB
ifig9_lowE_bias.html 0.0 MB
